In [ ]:
# Step 1: Import libraries and load data
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the dataset
path = os.path.join("data", "Dataset_AM_final.csv") 
data = pd.read_csv(path, sep=',', parse_dates=['date'])
data.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming train_df, val_df, test_df already exist in the environment

train_campaigns = set(train_df['traffic_source_campaign_name_anon'].unique())
val_campaigns   = set(val_df['traffic_source_campaign_name_anon'].unique())
test_campaigns  = set(test_df['traffic_source_campaign_name_anon'].unique())

# Prepare overlap counts
data = {
    "Set": ["Train", "Validation", "Test",
            "Train ∩ Validation", "Train ∩ Test", "Validation ∩ Test",
            "Train ∩ Val ∩ Test"],
    "Count": [
        len(train_campaigns),
        len(val_campaigns),
        len(test_campaigns),
        len(train_campaigns & val_campaigns),
        len(train_campaigns & test_campaigns),
        len(val_campaigns & test_campaigns),
        len(train_campaigns & val_campaigns & test_campaigns),
    ]
}

df_counts = pd.DataFrame(data)

# Plot simple bar chart
plt.figure(figsize=(10,5))
plt.bar(df_counts["Set"], df_counts["Count"])
plt.xticks(rotation=45, ha='right')
plt.title("Campaign Overlap Between Train / Validation / Test")
plt.ylabel("Number of Campaigns")
plt.tight_layout()
plt.show()

df_counts

In [ ]:
import matplotlib.pyplot as plt

# Build durations
duration = (
    data.groupby('traffic_source_campaign_name_anon')['date']
    .agg(['min', 'max'])
    .reset_index()
    .sort_values('min')
    .reset_index(drop=True)
)

plt.figure(figsize=(14, 14))

# plot each campaign as a thin line
for i, row in duration.iterrows():
    plt.plot([row['min'], row['max']], [i, i], color='black', linewidth=0.6)

# overlay train/val/test
plt.axvspan(train_df['date'].min(), train_df['date'].max(), color='green', alpha=0.10)
plt.axvspan(val_df['date'].min(),   val_df['date'].max(),   color='blue',  alpha=0.10)
plt.axvspan(test_df['date'].min(),  test_df['date'].max(),  color='red',   alpha=0.10)

plt.yticks([])  # hide vertical labels
plt.xlabel("Date")
plt.title("Campaign Activity Timeline (2000+ campaigns)")
plt.tight_layout()
plt.show()


In [ ]:
duration['days'] = (duration['max'] - duration['min']).dt.days + 1

plt.figure(figsize=(10,5))
plt.hist(duration['days'], bins=50)
MIN_CAMPAIGN_DURATION = 30
plt.axvline(MIN_CAMPAIGN_DURATION, color='red', linestyle='--', linewidth=2)
plt.title("Distribution of Campaign Durations")
plt.xlabel("Days active")
plt.ylabel("Number of campaigns")
plt.show()

In [ ]:
MIN_LEN = 30
duration['too_short'] = duration['days'] < MIN_LEN

def pct_short(split_set):
    split_campaigns = duration[duration['traffic_source_campaign_name_anon'].isin(split_set)]
    if len(split_campaigns) == 0:
        return 0
    return (split_campaigns['too_short'].sum() / len(split_campaigns)) * 100

print("Percentage of campaigns too short to form full sequences:")
print(f"Train: {pct_short(train_camps):.1f}%")
print(f"Validation: {pct_short(val_camps):.1f}%")
print(f"Test: {pct_short(test_camps):.1f}%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

MIN_LEN = 50
duration['too_short'] = duration['days'] < MIN_LEN

# Merge split info
duration['split'] = 'Other'
duration.loc[duration['traffic_source_campaign_name_anon'].isin(train_camps), 'split'] = 'Train'
duration.loc[duration['traffic_source_campaign_name_anon'].isin(val_camps),   'split'] = 'Validation'
duration.loc[duration['traffic_source_campaign_name_anon'].isin(test_camps),  'split'] = 'Test'

plt.figure(figsize=(12,6))

# Custom color palette
palette = {'Train':'#1f77b4', 'Validation':'#ff7f0e', 'Test':'#2ca02c', 'Other':'#d3d3d3'}
hue_order = ['Train', 'Validation', 'Test', 'Other']

sns.histplot(
    data=duration,
    x='days',
    hue='split',
    hue_order=hue_order,
    element='step',
    stat='percent',
    common_norm=False,
    bins=30,
    alpha=0.6,
    palette=palette
)

plt.axvline(MIN_LEN, color='red', linestyle='--', label=f'Min sequence length = {MIN_LEN}')
plt.xlabel('Campaign Duration (days)')
plt.ylabel('Percentage of campaigns')
plt.title('Campaign Duration Distribution by Split')
plt.legend(title='Split')  # force legend
plt.show()
